In [42]:
# Install required packages
!pip install networkx scikit-learn pandas numpy -q

import numpy as np
import pandas as pd
from typing import Dict, List, Set, Tuple, Optional, Any
import networkx as nx
from dataclasses import dataclass
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

In [43]:
import pandas as pd
import numpy as np

np.random.seed(42)

N = 1000  # number of rows

customers = pd.DataFrame({
    "customer_id": np.arange(1, N + 1),
    "age": np.random.randint(18, 70, N),
    "income": np.random.randint(30000, 130000, N),
    "past_purchases": np.random.poisson(8, N),
    "avg_order_value": np.round(np.random.uniform(20, 250, N), 2),
    "missing_fields": np.random.randint(1, 4, N)
})

# Ground truth label (valuable row)
customers["label_worth"] = (
    (customers["income"] > 70000) &
    (customers["past_purchases"] > 7) &
    (customers["avg_order_value"] > 80)
).astype(int)

# Acquisition cost
customers["acquisition_cost"] = (
    customers["missing_fields"] * np.random.randint(2, 6, N)
)

customers.to_csv("customers_partial.csv", index=False)

print("customers_partial.csv created")


customers_partial.csv created


Data Models

In [44]:
# ==================== DATA MODELS ====================
class DataQuality:
    HIGH = "high"
    MEDIUM = "medium"
    LOW = "low"
    ADVERSARIAL = "adversarial"

@dataclass
class AcquisitionOption:
    id: int
    rows: Set[int]
    columns: Set[str]
    cost: float
    base_utility: float
    quality: str = DataQuality.HIGH
    schema: Optional[Dict] = None
    table_id: Optional[str] = None
    confidence: float = 1.0

    def adjusted_utility(self, quality_weights: Dict[str, float]) -> float:
        """Calculate adjusted utility based on quality"""
        quality_weight = quality_weights.get(self.quality, 0.5)
        return self.base_utility * quality_weight * self.confidence

    @property
    def coverage(self) -> int:
        """Number of cells covered by this option"""
        return len(self.rows) * max(1, len(self.columns))

Dataset Creation

In [45]:
# ==================== DATASET CREATION & LOADING ====================

import pandas as pd
import numpy as np

np.random.seed(42)

N = 1000

# Main customer table
customers = pd.DataFrame({
    "row_id": np.arange(N),
    "age": np.random.randint(18, 70, N),
    "income": np.random.randint(30000, 130000, N),
    "past_purchases": np.random.poisson(8, N),
    "avg_order_value": np.round(np.random.uniform(20, 250, N), 2),
    "missing_fields": np.random.randint(1, 4, N)
})

# Ground truth utility (used only for evaluation)
customers["label_worth"] = (
    (customers["income"] > 70000) &
    (customers["past_purchases"] > 7) &
    (customers["avg_order_value"] > 80)
).astype(int)

# Acquisition cost per row
customers["acquisition_cost"] = (
    customers["missing_fields"] * np.random.randint(2, 6, N)
)

# Features and target
FEATURES = [
    "age",
    "income",
    "past_purchases",
    "avg_order_value",
    "missing_fields"
]
TARGET = "label_worth"

X = customers[FEATURES]
y = customers[TARGET]

print("Dataset ready:", customers.shape)


Dataset ready: (1000, 8)


Conformal Risk Control

In [46]:
class ConformalRiskControl:
    """Implements Conformal Risk Control for probabilistic confidence guarantees"""

    def __init__(self, risk_tolerance: float = 0.1, random_state: int = 42):
        self.risk_tolerance = risk_tolerance
        self.random_state = random_state
        self.lambda_star = None  # Optimal threshold
        self.scores = None
        self.labels = None

    def compute_scores(self, X: np.ndarray, y: np.ndarray, model: Any) -> np.ndarray:
        """Compute conformity scores - FIXED VERSION"""
        if hasattr(model, 'predict_proba'):
            probas = model.predict_proba(X)
            # FIX: Handle both binary and multiclass properly
            if probas.shape[1] == 2:  # Binary classification
                # For binary classification, use probability of class 1
                true_class_probs = probas[:, 1]
            else:
                # For multiclass, get probability of true class
                # Ensure y is integer type for indexing
                y_int = y.astype(int)
                true_class_probs = probas[np.arange(len(y_int)), y_int]
            scores = 1 - true_class_probs
        else:
            predictions = model.predict(X)
            scores = (predictions != y).astype(float)
        return scores

    def compute_threshold(self, X_calib: np.ndarray, y_calib: np.ndarray, model: Any) -> float:
        """Compute optimal threshold lambda_star"""
        scores = self.compute_scores(X_calib, y_calib, model)
        self.scores = scores
        self.labels = y_calib

        # Find lambda_star that controls risk
        lambda_star = 1.0
        for lam in np.linspace(0, 1, 100):
            predictions = (scores <= lam).astype(int)
            # Ensure y_calib is integer for comparison
            y_calib_int = y_calib.astype(int)
            risk = np.mean(predictions != y_calib_int)
            if risk <= self.risk_tolerance:
                lambda_star = lam
                break

        self.lambda_star = lambda_star
        return lambda_star

    def predict_with_control(self, X: np.ndarray, model: Any) -> Tuple[np.ndarray, np.ndarray]:
        """Predict with conformal risk control guarantees"""
        if self.lambda_star is None:
            raise ValueError("Must call compute_threshold first")

        # Create dummy y array of zeros for score computation
        dummy_y = np.zeros(len(X), dtype=int)
        scores = self.compute_scores(X, dummy_y, model)

        predictions = (scores <= self.lambda_star).astype(int)
        confidence_scores = 1 - scores

        return predictions, confidence_scores

RGREEDY Policy

In [47]:
class RGreedyAcquisition:
    """Implements RGreedy policy for row selection based on utility/cost ratio"""

    def __init__(self, budget: float, use_crc: bool = False,
                 crc_risk_tolerance: float = 0.1):
        self.budget = budget
        self.use_crc = use_crc
        self.crc = ConformalRiskControl(risk_tolerance=crc_risk_tolerance) if use_crc else None
        self.selection_model = None

    def train_selection_model(self, X_train: np.ndarray, y_train: np.ndarray,
                              utilities: np.ndarray, costs: np.ndarray):
        """Train a model to predict which rows are worth acquiring"""
        # Calculate utility/cost ratios
        ratios = utilities / np.maximum(costs, 1e-10)
        threshold = np.median(ratios)

        # Create labels: 1 if ratio > median, 0 otherwise
        labels = (ratios > threshold).astype(int)

        # Train the selection model
        self.selection_model = RandomForestClassifier(n_estimators=50, random_state=42, max_depth=5)
        self.selection_model.fit(X_train, labels)

        # If using CRC, calibrate threshold
        if self.use_crc and self.crc:
            # Split data for calibration
            X_train_sub, X_calib, y_train_sub, y_calib = train_test_split(
                X_train, labels, test_size=0.3, random_state=42
            )
            # Retrain on subset
            self.selection_model.fit(X_train_sub, y_train_sub)
            # Calibrate CRC threshold
            self.crc.compute_threshold(X_calib, y_calib, self.selection_model)

        return self.selection_model

    def select_rows(self, X: np.ndarray, predicted_utilities: np.ndarray,
                    costs: np.ndarray, row_indices: np.ndarray) -> List[int]:
        """Select rows using RGreedy policy"""
        if self.selection_model is None:
            return self._simple_greedy(predicted_utilities, costs, row_indices)

        # Get predictions
        if self.use_crc and self.crc:
            predictions, confidences = self.crc.predict_with_control(X, self.selection_model)
            # Weight utilities by confidence scores
            weighted_utilities = predicted_utilities * confidences
        else:
            predictions = self.selection_model.predict(X)
            weighted_utilities = predicted_utilities

        # Only consider rows predicted as worth acquiring (prediction = 1)
        valid_mask = predictions == 1
        if not valid_mask.any():
            return []

        # Filter valid rows
        valid_utilities = weighted_utilities[valid_mask]
        valid_costs = costs[valid_mask]
        valid_indices = row_indices[valid_mask]

        # Calculate utility/cost ratios for valid rows
        ratios = valid_utilities / np.maximum(valid_costs, 1e-10)

        # Sort by ratio (descending)
        sort_idx = np.argsort(-ratios)

        # Greedy selection within budget
        selected = []
        remaining_budget = self.budget

        for idx in sort_idx:
            if valid_costs[idx] <= remaining_budget:
                selected.append(int(valid_indices[idx]))
                remaining_budget -= valid_costs[idx]
                if remaining_budget <= 0:
                    break

        return selected

    def _simple_greedy(self, utilities: np.ndarray, costs: np.ndarray,
                       indices: np.ndarray) -> List[int]:
        """Simple greedy selection based on utility/cost ratio"""
        # Calculate ratios
        ratios = utilities / np.maximum(costs, 1e-10)

        # Sort by ratio (descending)
        sort_idx = np.argsort(-ratios)

        # Select within budget
        selected = []
        remaining_budget = self.budget

        for idx in sort_idx:
            if costs[idx] <= remaining_budget:
                selected.append(int(indices[idx]))
                remaining_budget -= costs[idx]
                if remaining_budget <= 0:
                    break

        return selected

CMOS Algorithm

In [48]:
class CMOSSelector:
    """Coverage Minimum Option Selection (CMOS)"""

    def __init__(self, overlap_penalty: float = 0.3, coverage_weight: float = 0.7):
        self.overlap_penalty = overlap_penalty
        self.coverage_weight = coverage_weight
        self.graph = nx.Graph()

    def build_option_graph(self, options: List[AcquisitionOption]) -> nx.Graph:
        """Build graph where nodes are options and edges represent overlaps"""
        self.graph.clear()

        # Add nodes with attributes
        for i, option in enumerate(options):
            self.graph.add_node(i,
                              option=option,
                              utility=option.base_utility,
                              cost=option.cost,
                              coverage=option.coverage)

        # Add edges for overlaps between options
        for i in range(len(options)):
            for j in range(i + 1, len(options)):
                overlap = self._compute_overlap(options[i], options[j])
                if overlap > 0:
                    # Edge weight represents the penalty for overlap
                    weight = overlap * self.overlap_penalty
                    self.graph.add_edge(i, j, weight=weight, overlap=overlap)

        return self.graph

    def _compute_overlap(self, option1: AcquisitionOption,
                        option2: AcquisitionOption) -> float:
        """Compute overlap between two options (0 to 1)"""
        if option1.table_id != option2.table_id:
            return 0.0

        # Compute row overlap percentage
        row_overlap = 0
        if option1.rows and option2.rows:
            row_overlap = len(option1.rows.intersection(option2.rows)) / max(
                len(option1.rows), len(option2.rows), 1
            )

        # Compute column overlap percentage
        col_overlap = 0
        if option1.columns and option2.columns:
            col_overlap = len(option1.columns.intersection(option2.columns)) / max(
                len(option1.columns), len(option2.columns), 1
            )

        # Return average overlap
        return (row_overlap + col_overlap) / 2

    def select_independent_set(self, budget: float) -> List[int]:
        """Select maximum utility independent set within budget"""
        if len(self.graph) == 0:
            return []

        # Initialize
        selected = []
        remaining_budget = budget

        # Get all nodes with their data
        nodes = list(self.graph.nodes(data=True))

        # Calculate adjusted utilities for each node
        node_scores = []
        for i, data in nodes:
            utility = data['utility']
            cost = data['cost']
            coverage = data['coverage']

            # Adjust utility by coverage (more coverage = better)
            coverage_bonus = 1 + self.coverage_weight * np.log1p(coverage)
            adjusted_utility = utility * coverage_bonus

            # Calculate initial score (utility/cost ratio)
            initial_score = adjusted_utility / max(cost, 0.01)

            node_scores.append((i, initial_score, adjusted_utility, cost))

        # Sort by initial score (descending)
        node_scores.sort(key=lambda x: x[1], reverse=True)

        # Greedy selection
        for node_id, _, adjusted_utility, cost in node_scores:
            if node_id in selected:
                continue

            if cost <= remaining_budget:
                # Check if this node is independent (no edges to already selected nodes)
                independent = True
                for selected_node in selected:
                    if self.graph.has_edge(node_id, selected_node):
                        independent = False
                        break

                if independent:
                    selected.append(node_id)
                    remaining_budget -= cost

            if remaining_budget <= 0:
                break

        return selected

Main CDA Network

In [49]:
class AdvancedCDAFramework:
    """
    Main framework integrating all components:
    - Conformal Risk Control (CRC)
    - RGreedy Policy
    - CMOS Algorithm
    - Budget Management
    """

    def __init__(self,
                 total_budget: float = 1000.0,
                 use_crc: bool = True,
                 crc_risk_tolerance: float = 0.1,
                 overlap_penalty: float = 0.3,
                 coverage_weight: float = 0.7):

        self.total_budget = total_budget
        self.remaining_budget = total_budget

        # Initialize components
        # Allocate 70% of budget for row acquisition, 30% for options
        self.rgreedy = RGreedyAcquisition(
            budget=total_budget * 0.7,
            use_crc=use_crc,
            crc_risk_tolerance=crc_risk_tolerance
        )

        self.cmos = CMOSSelector(
            overlap_penalty=overlap_penalty,
            coverage_weight=coverage_weight
        )

        # Tracking
        self.acquired_rows = set()
        self.acquired_options = []
        self.acquisition_history = []

    def predict_row_utilities(self, X: np.ndarray,
                              model: Optional[Any] = None) -> np.ndarray:
        """Predict utilities for rows using model or heuristic"""
        if model is not None and hasattr(model, 'predict_proba'):
            try:
                probas = model.predict_proba(X)
                if probas.shape[1] == 2:
                    # Binary classification - use uncertainty (entropy)
                    p = probas[:, 1]
                    # Avoid log(0) issues
                    p = np.clip(p, 1e-10, 1 - 1e-10)
                    entropy = -p * np.log(p) - (1 - p) * np.log(1 - p)
                    # Normalize to [0, 1]
                    utilities = entropy / np.log(2)
                else:
                    # Multiclass - use max probability uncertainty
                    max_probs = np.max(probas, axis=1)
                    utilities = 1 - max_probs
            except:
                # Fallback if prediction fails
                utilities = np.random.rand(len(X)) * 0.5 + 0.3
        else:
            # Heuristic: higher utility for rows with more variance
            if len(X) > 0:
                row_variance = np.var(X, axis=1)
                if np.max(row_variance) > 0:
                    utilities = row_variance / np.max(row_variance)
                else:
                    utilities = np.random.rand(len(X))
            else:
                utilities = np.array([])

        return utilities

    def acquire_rows(self,
                    dataset: pd.DataFrame,
                    costs: np.ndarray,
                    model: Optional[Any] = None,
                    table_id: str = "default") -> Set[int]:
        """Acquire rows using RGreedy policy with CRC"""
        if self.remaining_budget <= 0:
            return set()

        # Ensure costs array is the right length
        if len(costs) != len(dataset):
            raise ValueError(f"Costs array length {len(costs)} doesn't match dataset length {len(dataset)}")

        # Extract features
        X = dataset.values

        # Predict utilities for each row
        utilities = self.predict_row_utilities(X, model)

        # If this is our first acquisition, train the selection model
        if self.rgreedy.selection_model is None:
            # Create training data: label rows as "worth acquiring" if utility > median
            n_train = min(200, len(X))
            if n_train > 0:
                X_train = X[:n_train]
                y_train = (utilities[:n_train] > np.median(utilities[:n_train])).astype(int)

                self.rgreedy.train_selection_model(
                    X_train, y_train,
                    utilities[:n_train], costs[:n_train]
                )

        # Select rows using RGreedy
        row_indices = np.arange(len(dataset))
        selected_indices = self.rgreedy.select_rows(
            X, utilities, costs, row_indices
        )

        # Calculate total cost of selected rows
        total_cost = sum(costs[idx] for idx in selected_indices)

        # Check if we can afford this acquisition
        if total_cost <= self.remaining_budget:
            self.remaining_budget -= total_cost
            self.acquired_rows.update(selected_indices)

            # Record this acquisition
            self.acquisition_history.append({
                'type': 'row',
                'table_id': table_id,
                'indices': selected_indices.copy(),
                'cost': total_cost,
                'count': len(selected_indices),
                'avg_utility': np.mean(utilities[selected_indices]) if selected_indices else 0
            })

            return set(selected_indices)

        return set()

    def acquire_options(self, options: List[AcquisitionOption]) -> List[int]:
        """Acquire options using CMOS algorithm"""
        if self.remaining_budget <= 0:
            return []

        # Build the option graph
        self.cmos.build_option_graph(options)

        # Select an independent set of options
        selected_option_indices = self.cmos.select_independent_set(self.remaining_budget)

        # Calculate total cost and select options
        total_cost = 0
        selected_options = []

        for idx in selected_option_indices:
            option = options[idx]
            if option.cost <= (self.remaining_budget - total_cost):
                total_cost += option.cost
                selected_options.append(idx)
                self.acquired_options.append(option)
            else:
                break

        # Update budget if we can afford it
        if total_cost <= self.remaining_budget:
            self.remaining_budget -= total_cost

            # Record this acquisition
            self.acquisition_history.append({
                'type': 'option',
                'indices': selected_options.copy(),
                'cost': total_cost,
                'count': len(selected_options),
                'avg_utility': np.mean([options[i].base_utility for i in selected_options]) if selected_options else 0
            })

        return selected_options

    def get_acquisition_summary(self) -> Dict:
        """Get summary of all acquisitions"""
        total_rows = len(self.acquired_rows)
        total_options = len(self.acquired_options)
        total_spent = self.total_budget - self.remaining_budget

        # Calculate average utilities
        avg_row_utility = 0
        if total_rows > 0 and hasattr(self, 'last_utilities'):
            acquired_indices = list(self.acquired_rows)
            if max(acquired_indices) < len(self.last_utilities):
                avg_row_utility = np.mean(self.last_utilities[list(self.acquired_rows)])

        avg_option_utility = 0
        if total_options > 0:
            avg_option_utility = np.mean([opt.base_utility for opt in self.acquired_options])

        return {
            'total_budget': self.total_budget,
            'remaining_budget': self.remaining_budget,
            'total_spent': total_spent,
            'total_rows_acquired': total_rows,
            'total_options_acquired': total_options,
            'average_row_utility': avg_row_utility,
            'average_option_utility': avg_option_utility,
            'budget_utilization': (total_spent / self.total_budget) * 100 if self.total_budget > 0 else 0,
            'acquisition_history': self.acquisition_history
        }

Demo Functions

In [50]:
def create_sample_dataset(n_samples: int = 500, n_features: int = 8):
    """Create a sample dataset for testing"""
    # Set random seed for reproducibility
    np.random.seed(42)

    # Create feature matrix
    features = np.random.randn(n_samples, n_features)

    # Create DataFrame
    data = pd.DataFrame(
        features,
        columns=[f'feature_{i}' for i in range(n_features)]
    )

    # Add some categorical features
    data['category'] = np.random.choice(['A', 'B', 'C'], n_samples)

    # Add target variable (binary classification)
    data['target'] = (np.random.rand(n_samples) > 0.5).astype(int)

    # Create costs - some rows are more expensive than others
    base_costs = np.random.exponential(5, n_samples) + 1  # Base cost $1-20
    complexity_penalty = np.abs(features).sum(axis=1) * 0.5  # More complex rows cost more
    costs = base_costs + complexity_penalty

    # Create acquisition options
    options = []
    for i in range(5):
        # Randomly select rows for this option
        n_rows = np.random.randint(30, min(150, n_samples))
        rows = set(np.random.choice(n_samples, n_rows, replace=False))

        # Randomly select columns
        n_cols = np.random.randint(2, min(5, n_features + 1))
        available_cols = list(data.columns[:-2])  # Exclude 'category' and 'target'
        columns = set(np.random.choice(available_cols, n_cols, replace=False))

        # Create option
        option = AcquisitionOption(
            id=i,
            rows=rows,
            columns=columns,
            cost=np.random.uniform(30, 300),
            base_utility=np.random.uniform(0.3, 0.95),
            quality=np.random.choice([DataQuality.HIGH, DataQuality.MEDIUM,
                                     DataQuality.LOW], p=[0.7, 0.2, 0.1]),
            table_id="sample_table"
        )
        options.append(option)

    return {
        'data': data,
        'costs': costs,
        'options': options
    }

def run_demo():
    """Run a complete demonstration of the CDA framework"""
    print("=" * 70)
    print("COMPLETE CDA FRAMEWORK DEMONSTRATION")
    print("=" * 70)

    # Step 1: Create sample dataset
    print("\n📊 1. Creating sample dataset...")
    sample_data = create_sample_dataset(n_samples=300, n_features=6)
    data = sample_data['data']
    costs = sample_data['costs']
    options = sample_data['options']

    print(f"   • Dataset shape: {data.shape}")
    print(f"   • Number of acquisition options: {len(options)}")
    print(f"   • Total budget: $1000.00")
    print(f"   • Cost range: ${costs.min():.2f} - ${costs.max():.2f}")

    # Step 2: Initialize framework
    print("\n🚀 2. Initializing CDA Framework...")
    framework = AdvancedCDAFramework(
        total_budget=1000.0,
        use_crc=True,
        crc_risk_tolerance=0.1,
        overlap_penalty=0.2
    )
    print("   • Framework initialized with CRC enabled")

    # Step 3: Prepare data and train a model
    print("\n🤖 3. Training utility prediction model...")
    # Separate features and target
    X = data.drop(['target', 'category'], axis=1)
    y = data['target']

    # One-hot encode categorical variables
    X_encoded = pd.get_dummies(data.drop('target', axis=1), columns=['category'])

    # Train a simple Random Forest model
    model = RandomForestClassifier(n_estimators=30, random_state=42, max_depth=5)
    model.fit(X_encoded, y)
    print(f"   • Model trained on {len(X)} samples")
    print(f"   • Model accuracy: {model.score(X_encoded, y):.3f}")

    # Step 4: Acquire rows using RGreedy with CRC
    print("\n💰 4. Acquiring rows using RGreedy with Conformal Risk Control...")
    acquired_rows = framework.acquire_rows(
        dataset=X_encoded,
        costs=costs,
        model=model,
        table_id="sample_table"
    )
    print(f"   • Acquired {len(acquired_rows)} rows")
    if acquired_rows:
        avg_cost = np.mean(costs[list(acquired_rows)])
        print(f"   • Average cost per row: ${avg_cost:.2f}")

    # Step 5: Acquire options using CMOS
    print("\n🔄 5. Acquiring options using CMOS algorithm...")
    selected_options = framework.acquire_options(options)
    print(f"   • Selected {len(selected_options)} options")
    if selected_options:
        total_option_cost = sum(options[i].cost for i in selected_options)
        avg_option_utility = np.mean([options[i].base_utility for i in selected_options])
        print(f"   • Total option cost: ${total_option_cost:.2f}")
        print(f"   • Average option utility: {avg_option_utility:.3f}")

    # Step 6: Show acquisition summary
    print("\n📈 6. Acquisition Summary:")
    summary = framework.get_acquisition_summary()

    print(f"   • Total Budget: ${summary['total_budget']:.2f}")
    print(f"   • Remaining Budget: ${summary['remaining_budget']:.2f}")
    print(f"   • Total Spent: ${summary['total_spent']:.2f}")
    print(f"   • Rows Acquired: {summary['total_rows_acquired']}")
    print(f"   • Options Acquired: {summary['total_options_acquired']}")
    print(f"   • Budget Utilization: {summary['budget_utilization']:.1f}%")

    # Step 7: Show some statistics
    print("\n📊 7. Performance Statistics:")
    if acquired_rows:
        acquired_data = data.iloc[list(acquired_rows)]
        print(f"   • Acquired data shape: {acquired_data.shape}")
        print(f"   • Target distribution in acquired data:")
        target_counts = acquired_data['target'].value_counts()
        for val, count in target_counts.items():
            percentage = count / len(acquired_data) * 100
            print(f"     - Class {val}: {count} samples ({percentage:.1f}%)")

    print("\n" + "=" * 70)
    print("✅ DEMO COMPLETED SUCCESSFULLY!")
    print("=" * 70)

    return framework, summary

Main Execution

In [51]:
if __name__ == "__main__":
    print("🔧 Installing and importing required packages...")

    # Run the demo
    try:
        framework, summary = run_demo()

        # Show example of how to use the framework
        print("\n" + "=" * 70)
        print("💡 HOW TO USE THIS FRAMEWORK:")
        print("=" * 70)

        print("""
# 1. Initialize the framework with your budget
framework = AdvancedCDAFramework(
    total_budget=5000.0,          # Your total budget
    use_crc=True,                 # Enable Conformal Risk Control
    crc_risk_tolerance=0.05,      # 5% risk tolerance
    overlap_penalty=0.3           # Penalty for overlapping options
)

# 2. Load your data
import pandas as pd
import numpy as np

# Your dataset
data = pd.read_csv('your_data.csv')

# Define costs for each row (could be based on data complexity, etc.)
costs = calculate_costs(data)  # Your cost function

# 3. (Optional) Train a model for utility prediction
from sklearn.ensemble import RandomForestClassifier

# Prepare features and target
X = data.drop('target_column', axis=1)
y = data['target_column']

# Train model
model = RandomForestClassifier(n_estimators=100)
model.fit(X, y)

# 4. Acquire rows
acquired_rows = framework.acquire_rows(
    dataset=X,
    costs=costs,
    model=model,          # Optional
    table_id="your_table"
)

# 5. Create acquisition options (if you have them)
options = [
    AcquisitionOption(
        id=0,
        rows={0, 1, 2, 3},      # Set of row indices
        columns={'col1', 'col2'}, # Set of column names
        cost=150.0,              # Cost of this option
        base_utility=0.85,       # Estimated utility
        quality=DataQuality.HIGH # Data quality
    ),
    # ... more options
]

# 6. Acquire options
selected_options = framework.acquire_options(options)

# 7. Get results
summary = framework.get_acquisition_summary()
print(f"Budget spent: ${summary['total_spent']:.2f}")
print(f"Rows acquired: {summary['total_rows_acquired']}")
print(f"Budget utilization: {summary['budget_utilization']:.1f}%")
        """)

    except Exception as e:
        print(f"\n❌ Error during execution: {e}")
        print("\nTrying simplified version...")

        # Try a simpler version if the full demo fails
        try:
            # Simple test
            print("\nRunning simplified test...")
            from sklearn.ensemble import RandomForestClassifier

            # Create tiny dataset
            X_simple = pd.DataFrame({'f1': [1,2,3,4,5], 'f2': [10,20,30,40,50]})
            costs_simple = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
            y_simple = np.array([0, 1, 0, 1, 0])

            model_simple = RandomForestClassifier(n_estimators=10, random_state=42)
            model_simple.fit(X_simple, y_simple)

            framework_simple = AdvancedCDAFramework(total_budget=10.0, use_crc=False)
            acquired = framework_simple.acquire_rows(X_simple, costs_simple, model_simple)

            print(f"Simple test acquired {len(acquired)} rows: {acquired}")
            print("✅ Simplified test passed!")

        except Exception as e2:
            print(f"❌ Simplified test also failed: {e2}")

🔧 Installing and importing required packages...
COMPLETE CDA FRAMEWORK DEMONSTRATION

📊 1. Creating sample dataset...
   • Dataset shape: (300, 8)
   • Number of acquisition options: 5
   • Total budget: $1000.00
   • Cost range: $2.15 - $43.12

🚀 2. Initializing CDA Framework...
   • Framework initialized with CRC enabled

🤖 3. Training utility prediction model...
   • Model trained on 300 samples
   • Model accuracy: 0.877

💰 4. Acquiring rows using RGreedy with Conformal Risk Control...
   • Acquired 141 rows
   • Average cost per row: $4.96

🔄 5. Acquiring options using CMOS algorithm...
   • Selected 1 options
   • Total option cost: $68.10
   • Average option utility: 0.655

📈 6. Acquisition Summary:
   • Total Budget: $1000.00
   • Remaining Budget: $232.09
   • Total Spent: $767.91
   • Rows Acquired: 141
   • Options Acquired: 1
   • Budget Utilization: 76.8%

📊 7. Performance Statistics:
   • Acquired data shape: (141, 8)
   • Target distribution in acquired data:
     - Clas